# Temporal Recommendation Benchmark — Final

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Imports and benchmark configuration

The learner split intentionally follows the Phase 1 convention: a chronological 70/30 split within every eligible learner, followed by an 80/20 discovery/validation learner split.

In [2]:
from pathlib import Path
import gc
import json
import warnings

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore', category=FutureWarning)

RANDOM_STATE = 42
EARLY_FRACTION = 0.70
VALIDATION_FRACTION = 0.20
MIN_INTERACTIONS = 20
MASTERY_PRIOR_STRENGTH = 5.0
MIN_MASTERY_EVIDENCE = 3
WEAK_MASTERY_THRESHOLD = 0.60
MASTERED_MASTERY_THRESHOLD = 0.80

ASSIST_DATA_PATH = '/content/drive/MyDrive/datasets/2012-2013-data-with-predictions-4-final.csv'
KDD_DATA_PATH = '/content/drive/MyDrive/datasets/algebra_2005_2006_train.txt'
ASSIST_CLUSTER_DIR = Path('/content/drive/MyDrive/datasets/assistments_final_outputs')
KDD_CLUSTER_DIR = Path('/content/drive/MyDrive/datasets/kdd_final_outputs')
OUTPUT_ROOT = Path('/content/drive/MyDrive/datasets/recommendation_benchmark_final_outputs')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

CATALOG_THRESHOLDS = {
    'ASSISTments': {'skill': 100, 'problem': 20},
    'KDD': {'skill': 10, 'problem': 5},
}
ROW_LIMITS = {'ASSISTments': None, 'KDD': None}

for required in [
    ASSIST_CLUSTER_DIR / 'learner_cluster_assignments.csv',
    ASSIST_CLUSTER_DIR / 'run_config.json',
    KDD_CLUSTER_DIR / 'learner_cluster_assignments.csv',
    KDD_CLUSTER_DIR / 'run_config.json',
]:
    if not required.exists():
        raise FileNotFoundError(f'Missing Phase 1 artifact: {required}')
print('Configuration and Phase 1 artifact contract validated.')

Configuration and Phase 1 artifact contract validated.


## Benchmark definitions and leakage controls

Each event receives a stable learner, event, problem and skill representation. Catalogues, popularity, difficulty and problem-skill associations are derived only from discovery-early data.

In [ ]:
def normalise_learner_ids(values):
    return (
        values.astype(str).str.strip()
        .str.replace(r'^(-?\d+)\.0$', r'\1', regex=True)
    )

def normalise_skill_names(values):
    return (
        values.fillna('').astype(str).str.strip().str.casefold()
        .str.replace(r'\s+', ' ', regex=True)
    )

def apply_assist_training_skill_mapping(early, future):
    early = early.copy()
    future = future.copy()
    early['normalised_skill_name'] = normalise_skill_names(early['skill_name'])
    future['normalised_skill_name'] = normalise_skill_names(future['skill_name'])
    invalid_names = {'', 'unknown', 'nan', 'none'}

    mapping_source = early[
        early['cohort'].eq('discovery')
        & early['skill_numeric_id'].notna()
        & ~early['normalised_skill_name'].isin(invalid_names)
    ].copy()
    skill_mapping = mapping_source.groupby('normalised_skill_name', sort=False).agg(
        mapped_skill_id=('skill_numeric_id', lambda values: values.mode().iloc[0]),
        training_rows=('event_id', 'size'),
        training_learners=('learner_id', 'nunique'),
        distinct_skill_ids=('skill_numeric_id', 'nunique'),
    ).reset_index()
    skill_mapping['mapped_skill_id'] = skill_mapping['mapped_skill_id'].astype('Int64')
    skill_mapping['mapping_source'] = 'discovery_early_only'
    mapping_lookup = skill_mapping.set_index('normalised_skill_name')['mapped_skill_id']

    def encode(frame):
        native_id = pd.to_numeric(frame['skill_numeric_id'], errors='coerce').astype('Float64')
        mapped_id = frame['normalised_skill_name'].map(mapping_lookup).astype('Float64')
        resolved_id = native_id.fillna(mapped_id)
        valid_text = (
            resolved_id.isna()
            & ~frame['normalised_skill_name'].isin(invalid_names)
        )
        skill_item = pd.Series(pd.NA, index=frame.index, dtype='object')
        has_id = resolved_id.notna()
        skill_item.loc[has_id] = (
            'skill:id:' + resolved_id.loc[has_id].astype('Int64').astype(str)
        )
        skill_item.loc[valid_text] = 'skill:text:' + frame.loc[valid_text, 'normalised_skill_name']
        frame['skill_items'] = [
            [] if pd.isna(item) else [item] for item in skill_item.to_numpy()
        ]
        frame['skill_identity_source'] = np.select(
            [
                native_id.notna(),
                native_id.isna() & mapped_id.notna(),
                valid_text,
            ],
            ['native_id', 'discovery_early_name_map', 'text_fallback'],
            default='unknown_excluded',
        )
        return frame

    early = encode(early)
    future = encode(future)
    diagnostics = {
        'TrainingSkillNameMappings': len(skill_mapping),
        'AmbiguousTrainingSkillNames': int((skill_mapping['distinct_skill_ids'] > 1).sum()),
        'EarlyRowsMappedFromName': int(early['skill_identity_source'].eq('discovery_early_name_map').sum()),
        'FutureRowsMappedFromName': int(future['skill_identity_source'].eq('discovery_early_name_map').sum()),
        'EarlyTextFallbackRows': int(early['skill_identity_source'].eq('text_fallback').sum()),
        'FutureTextFallbackRows': int(future['skill_identity_source'].eq('text_fallback').sum()),
    }
    assert skill_mapping['mapping_source'].eq('discovery_early_only').all()
    assert set(skill_mapping['normalised_skill_name']).issubset(
        set(mapping_source['normalised_skill_name'])
    )
    return early, future, skill_mapping, diagnostics

def chronological_split(events):
    events = events.sort_values(
        ['learner_id', 'event_time', 'event_id'], na_position='last'
    ).copy()
    counts = events.groupby('learner_id').size()
    eligible = counts[counts >= MIN_INTERACTIONS].index
    events = events[events['learner_id'].isin(eligible)].copy()
    events['event_number'] = events.groupby('learner_id').cumcount()
    events['learner_event_count'] = events.groupby('learner_id')['learner_id'].transform('size')
    raw_cutoff = np.floor(events['learner_event_count'] * EARLY_FRACTION).astype(int)
    events['split_point'] = np.minimum(
        np.maximum(raw_cutoff, 1), events['learner_event_count'] - 1
    )
    early = events[events['event_number'] < events['split_point']].copy()
    future = events[events['event_number'] >= events['split_point']].copy()
    assert early.index.intersection(future.index).empty
    assert early['learner_id'].nunique() == future['learner_id'].nunique()
    users = early['learner_id'].drop_duplicates().to_numpy()
    discovery, validation = train_test_split(
        users, test_size=VALIDATION_FRACTION, random_state=RANDOM_STATE
    )
    cohort = {user: 'discovery' for user in discovery}
    cohort.update({user: 'validation' for user in validation})
    early['cohort'] = early['learner_id'].map(cohort)
    future['cohort'] = future['learner_id'].map(cohort)
    return early, future, discovery, validation

def explode_skills(events):
    skill_events = events.explode('skill_items').rename(columns={'skill_items': 'item_id'})
    skill_events = skill_events[skill_events['item_id'].notna()].copy()
    skill_events['item_id'] = skill_events['item_id'].astype(str).str.strip()
    return skill_events[skill_events['item_id'].ne('')]

def build_catalog(training_events, item_column, item_type, minimum_learners):
    source = training_events[training_events[item_column].notna()].copy()
    source[item_column] = source[item_column].astype(str)
    catalog = source.groupby(item_column, sort=False).agg(
        training_interactions=('correct', 'size'),
        training_learners=('learner_id', 'nunique'),
        training_success_rate=('correct', 'mean'),
        training_mean_hints=('hints', 'mean'),
        training_mean_attempts=('attempts', 'mean'),
    ).reset_index().rename(columns={item_column: 'item_id'})
    catalog = catalog[catalog['training_learners'] >= minimum_learners].copy()
    catalog['item_type'] = item_type
    catalog['difficulty_proxy'] = 1 - catalog['training_success_rate']
    catalog['training_popularity'] = (
        catalog['training_interactions'] / catalog['training_interactions'].sum()
    )
    catalog['catalog_source'] = 'discovery_early_only'
    return catalog.sort_values(['training_learners', 'item_id'], ascending=[False, True]).reset_index(drop=True)

def aggregate_history(events, item_column, prefix):
    frame = events[events[item_column].notna()].copy()
    frame[item_column] = frame[item_column].astype(str)
    result = frame.groupby(['learner_id', item_column], sort=False).agg(
        interaction_count=('correct', 'size'),
        correct_count=('correct', 'sum'),
        success_rate=('correct', 'mean'),
        total_hints=('hints', 'sum'),
        mean_attempts=('attempts', 'mean'),
        first_event_number=('event_number', 'min'),
        last_event_number=('event_number', 'max'),
        first_event_time=('event_time', 'min'),
        last_event_time=('event_time', 'max'),
    ).reset_index().rename(columns={item_column: 'item_id'})
    metric_columns = [column for column in result.columns if column not in ['learner_id', 'item_id']]
    result = result.rename(columns={column: f'{prefix}_{column}' for column in metric_columns})
    result['learner_id'] = result['learner_id'].astype(str)
    return result

def add_skill_mastery(early_skill_history, skill_catalog):
    priors = skill_catalog[['item_id', 'training_success_rate']].rename(
        columns={'training_success_rate': 'discovery_early_skill_prior'}
    )
    history = early_skill_history.merge(priors, on='item_id', how='left')
    history['in_candidate_catalog'] = history['discovery_early_skill_prior'].notna()
    history['empirical_bayes_mastery'] = (
        history['early_correct_count']
        + MASTERY_PRIOR_STRENGTH * history['discovery_early_skill_prior']
    ) / (history['early_interaction_count'] + MASTERY_PRIOR_STRENGTH)
    history['mastery_evidence_confidence'] = (
        history['early_interaction_count']
        / (history['early_interaction_count'] + MASTERY_PRIOR_STRENGTH)
    )
    history['mastery_delta_from_skill_prior'] = (
        history['empirical_bayes_mastery'] - history['discovery_early_skill_prior']
    )
    history['skill_state'] = np.select(
        [
            ~history['in_candidate_catalog'],
            history['early_interaction_count'] < MIN_MASTERY_EVIDENCE,
            history['empirical_bayes_mastery'] <= WEAK_MASTERY_THRESHOLD,
            history['empirical_bayes_mastery'] >= MASTERED_MASTERY_THRESHOLD,
        ],
        ['out_of_catalog', 'insufficient_evidence', 'weak', 'mastered'],
        default='developing',
    )
    return history

def make_future_relevance(future_history, early_history, catalog):
    relevance = future_history.copy()
    future_count_column = next(column for column in relevance.columns if column.endswith('_interaction_count'))
    future_correct_column = next(column for column in relevance.columns if column.endswith('_correct_count'))
    relevance['relevance_binary'] = (relevance[future_count_column] >= 1).astype('int8')
    relevance['successful_future_item'] = (relevance[future_correct_column] >= 1).astype('int8')
    relevance['in_candidate_catalog'] = relevance['item_id'].isin(set(catalog['item_id']))
    seen = early_history[['learner_id', 'item_id']].drop_duplicates().assign(seen_in_early=True)
    relevance = relevance.merge(seen, on=['learner_id', 'item_id'], how='left')
    relevance['seen_in_early'] = relevance['seen_in_early'].fillna(False).astype(bool)
    return relevance

def load_cluster_labels(directory):
    assignments = pd.read_csv(directory / 'learner_cluster_assignments.csv')
    identifier_candidates = ['user_id', 'Anon Student Id', 'learner_id']
    identifier = next((column for column in identifier_candidates if column in assignments.columns), None)
    if identifier is None:
        unnamed = [column for column in assignments.columns if column.startswith('Unnamed:')]
        if not unnamed:
            raise ValueError(f'Cannot identify learner column in {directory}')
        identifier = unnamed[0]
    labels = assignments[[identifier, 'cluster']].copy()
    labels['learner_id'] = normalise_learner_ids(labels[identifier])
    with open(directory / 'run_config.json') as file:
        config = json.load(file)
    return labels[['learner_id', 'cluster']], config

def parquet_rows(path):
    return pq.ParquetFile(path).metadata.num_rows

## Common benchmark builder

The builder writes compact learner-item histories. Both all-future relevance and in-catalog indicators are retained so catalogue coverage and cold-start exclusions remain visible.

In [4]:
def build_benchmark(events, dataset, cluster_directory):
    raw_rows = len(events)
    raw_learners = events['learner_id'].nunique()
    early, future, discovery_users, validation_users = chronological_split(events)
    eligible_learners = early['learner_id'].nunique()

    # Cohort is assigned before IDs are normalised for portable storage.
    early['learner_id'] = normalise_learner_ids(early['learner_id'])
    future['learner_id'] = normalise_learner_ids(future['learner_id'])
    skill_mapping = pd.DataFrame({
        'normalised_skill_name': pd.Series(dtype='object'),
        'mapped_skill_id': pd.Series(dtype='Int64'),
        'training_rows': pd.Series(dtype='int64'),
        'training_learners': pd.Series(dtype='int64'),
        'distinct_skill_ids': pd.Series(dtype='int64'),
        'mapping_source': pd.Series(dtype='object'),
    })
    mapping_diagnostics = {
        'TrainingSkillNameMappings': 0, 'AmbiguousTrainingSkillNames': 0,
        'EarlyRowsMappedFromName': 0, 'FutureRowsMappedFromName': 0,
        'EarlyTextFallbackRows': 0, 'FutureTextFallbackRows': 0,
    }
    if dataset == 'ASSISTments':
        early, future, skill_mapping, mapping_diagnostics = apply_assist_training_skill_mapping(
            early, future
        )
    discovery_early = early[early['cohort'].eq('discovery')].copy()

    early_skills = explode_skills(early)
    future_skills = explode_skills(future)
    training_skills = explode_skills(discovery_early)

    thresholds = CATALOG_THRESHOLDS[dataset]
    skill_catalog = build_catalog(training_skills, 'item_id', 'skill', thresholds['skill'])
    problem_catalog = build_catalog(discovery_early, 'problem_item', 'problem', thresholds['problem'])

    # Problem metadata and problem-skill relations also use discovery-early only.
    problem_metadata = (
        discovery_early[['problem_item', 'problem_type', 'hierarchy']]
        .drop_duplicates('problem_item')
        .rename(columns={'problem_item': 'item_id'})
    )
    problem_catalog = problem_catalog.merge(problem_metadata, on='item_id', how='left')
    problem_skill_map = (
        training_skills.groupby(['problem_item', 'item_id'], sort=False).size()
        .rename('training_interactions').reset_index()
        .rename(columns={'problem_item': 'problem_item_id', 'item_id': 'skill_item_id'})
    )
    problem_skill_map = problem_skill_map[
        problem_skill_map['problem_item_id'].isin(set(problem_catalog['item_id']))
        & problem_skill_map['skill_item_id'].isin(set(skill_catalog['item_id']))
    ].copy()
    association_total = problem_skill_map.groupby('problem_item_id')['training_interactions'].transform('sum')
    problem_skill_map['association_share'] = problem_skill_map['training_interactions'] / association_total
    problem_skill_map['mapping_source'] = 'discovery_early_only'

    early_skill_history = aggregate_history(early_skills, 'item_id', 'early')
    early_skill_history = add_skill_mastery(early_skill_history, skill_catalog)
    future_skill_history = aggregate_history(future_skills, 'item_id', 'future')
    early_problem_history = aggregate_history(early, 'problem_item', 'early')
    future_problem_history = aggregate_history(future, 'problem_item', 'future')

    early_problem_history['in_candidate_catalog'] = early_problem_history['item_id'].isin(set(problem_catalog['item_id']))
    future_skill_relevance = make_future_relevance(future_skill_history, early_skill_history, skill_catalog)
    future_problem_relevance = make_future_relevance(future_problem_history, early_problem_history, problem_catalog)

    learner_splits = early.groupby(['learner_id', 'cohort'], sort=False).agg(
        early_interactions=('event_id', 'size'),
        early_unique_problems=('problem_item', 'nunique'),
        early_last_time=('event_time', 'max'),
    ).reset_index()
    future_counts = future.groupby('learner_id', sort=False).agg(
        future_interactions=('event_id', 'size'),
        future_unique_problems=('problem_item', 'nunique'),
        future_first_time=('event_time', 'min'),
    ).reset_index()
    learner_splits = learner_splits.merge(future_counts, on='learner_id', how='left')

    def add_item_counts(base, history, relevance, label):
        history_counts = history[history['in_candidate_catalog']].groupby('learner_id')['item_id'].nunique()
        future_counts = relevance[relevance['in_candidate_catalog']].groupby('learner_id')['item_id'].nunique()
        base[f'early_catalog_{label}s'] = base['learner_id'].map(history_counts).fillna(0).astype(int)
        base[f'future_catalog_{label}s'] = base['learner_id'].map(future_counts).fillna(0).astype(int)
        base[f'cold_start_{label}_history'] = base[f'early_catalog_{label}s'].eq(0)
        base[f'evaluable_{label}'] = base[f'future_catalog_{label}s'].gt(0)

    add_item_counts(learner_splits, early_skill_history, future_skill_relevance, 'skill')
    add_item_counts(learner_splits, early_problem_history, future_problem_relevance, 'problem')
    skill_state_counts = (
        early_skill_history[early_skill_history['in_candidate_catalog']]
        .groupby(['learner_id', 'skill_state']).size().unstack(fill_value=0)
    )
    for state in ['weak', 'developing', 'mastered', 'insufficient_evidence']:
        learner_splits[f'early_{state}_skills'] = (
            learner_splits['learner_id'].map(skill_state_counts.get(state, pd.Series(dtype=int)))
            .fillna(0).astype(int)
        )
    cluster_labels, cluster_config = load_cluster_labels(cluster_directory)
    learner_splits = learner_splits.merge(cluster_labels, on='learner_id', how='left', validate='one_to_one')
    learner_splits['cluster_selection_status'] = cluster_config.get('selected_solution_status', 'not_recorded')
    learner_splits['cluster_allowed_as_primary'] = bool(cluster_config.get('selection_passed_gates', False))

    # Explicit leakage and integrity assertions.
    assert set(skill_catalog['item_id']).issubset(set(training_skills['item_id']))
    assert set(problem_catalog['item_id']).issubset(set(discovery_early['problem_item']))
    assert skill_catalog['catalog_source'].eq('discovery_early_only').all()
    assert problem_catalog['catalog_source'].eq('discovery_early_only').all()
    assert learner_splits['cohort'].isin(['discovery', 'validation']).all()
    assert learner_splits['cluster'].notna().all()

    tables = {
        'learner_splits.parquet': learner_splits,
        'skill_catalog.parquet': skill_catalog,
        'problem_catalog.parquet': problem_catalog,
        'early_skill_history.parquet': early_skill_history,
        'future_skill_relevance.parquet': future_skill_relevance,
        'early_problem_history.parquet': early_problem_history,
        'future_problem_relevance.parquet': future_problem_relevance,
        'problem_skill_map.parquet': problem_skill_map,
        'skill_name_id_map.parquet': skill_mapping,
    }

    summary = {
        'Dataset': dataset,
        'RawRows': raw_rows,
        'RawLearners': raw_learners,
        'EligibleLearners': eligible_learners,
        'DiscoveryLearners': len(discovery_users),
        'ValidationLearners': len(validation_users),
        'EarlyInteractions': len(early),
        'FutureInteractions': len(future),
        'SkillCandidates': len(skill_catalog),
        'ProblemCandidates': len(problem_catalog),
        'ValidationSkillEvaluableRate': learner_splits.loc[learner_splits['cohort'].eq('validation'), 'evaluable_skill'].mean(),
        'ValidationProblemEvaluableRate': learner_splits.loc[learner_splits['cohort'].eq('validation'), 'evaluable_problem'].mean(),
        'ValidationSkillColdStartRate': learner_splits.loc[learner_splits['cohort'].eq('validation'), 'cold_start_skill_history'].mean(),
        'ValidationProblemColdStartRate': learner_splits.loc[learner_splits['cohort'].eq('validation'), 'cold_start_problem_history'].mean(),
        'ClusterSelectionStatus': cluster_config.get('selected_solution_status', 'not_recorded'),
        **mapping_diagnostics,
    }
    return tables, summary

## ASSISTments benchmark

ASSISTments is the primary recommendation dataset.

In [5]:
assist_columns = [
    'problem_log_id', 'user_id', 'problem_id', 'assignment_id', 'sequence_id',
    'skill', 'skill_id', 'problem_type', 'start_time', 'end_time',
    'correct', 'hint_count', 'attempt_count',
]
assist_raw = pd.read_csv(
    ASSIST_DATA_PATH, usecols=assist_columns, nrows=ROW_LIMITS['ASSISTments'], low_memory=False
)
for column in ['problem_log_id', 'user_id', 'problem_id', 'assignment_id', 'sequence_id', 'skill_id', 'correct', 'hint_count', 'attempt_count']:
    assist_raw[column] = pd.to_numeric(assist_raw[column], errors='coerce')
assist_raw['start_time'] = pd.to_datetime(assist_raw['start_time'], errors='coerce')
assist_raw['end_time'] = pd.to_datetime(assist_raw['end_time'], errors='coerce')
assist_raw['event_time'] = assist_raw['start_time'].fillna(assist_raw['end_time'])
duplicate_event = assist_raw['problem_log_id'].notna() & assist_raw.duplicated('problem_log_id', keep='first')
assist_raw = assist_raw.loc[~duplicate_event]
assist_raw = assist_raw[assist_raw['user_id'].notna() & assist_raw['correct'].isin([0, 1])].copy()
assist_raw['hints'] = assist_raw['hint_count'].clip(lower=0).fillna(0)
assist_raw['attempts'] = assist_raw['attempt_count'].clip(lower=0).fillna(1)

# Preserve raw skill identity until after chronological and learner splits.
# The discovery-early mapping is learned inside build_benchmark.
assist_raw['skill_name'] = assist_raw['skill']
assist_raw['skill_numeric_id'] = assist_raw['skill_id']
assist_raw['problem_item'] = 'problem:' + assist_raw['problem_id'].astype('Int64').astype(str)
assist_raw.loc[assist_raw['problem_id'].isna(), 'problem_item'] = pd.NA
assist_raw['hierarchy'] = (
    'assignment:' + assist_raw['assignment_id'].astype('Int64').astype(str)
    + '|sequence:' + assist_raw['sequence_id'].astype('Int64').astype(str)
)
assist_events = assist_raw.rename(columns={
    'user_id': 'learner_id', 'problem_log_id': 'event_id',
})[[
    'learner_id', 'event_id', 'event_time', 'problem_item',
    'skill_name', 'skill_numeric_id',
    'correct', 'hints', 'attempts', 'problem_type', 'hierarchy',
]].copy()
print('ASSISTments standardised events:', assist_events.shape)
print('Missing timestamps:', assist_events['event_time'].isna().sum())
assist_tables, assist_summary = build_benchmark(
    assist_events, 'ASSISTments', ASSIST_CLUSTER_DIR
)
display(pd.Series(assist_summary))

ASSISTments standardised events: (6117947, 11)
Missing timestamps: 0


,0
Dataset,ASSISTments
RawRows,6117947
RawLearners,46667
EligibleLearners,33335
DiscoveryLearners,26668
ValidationLearners,6667
EarlyInteractions,4184236
FutureInteractions,1814850
SkillCandidates,161
ProblemCandidates,39779


## Release ASSISTments event memory before loading KDD

In [6]:
del assist_raw, assist_events
gc.collect()
print('Raw ASSISTments event memory released; compact benchmark tables retained for saving.')

Raw ASSISTments event memory released; compact benchmark tables retained for saving.


## KDD benchmark

KDD is retained as secondary validation.

In [7]:
kdd_raw = pd.read_csv(
    KDD_DATA_PATH, sep='\t', nrows=ROW_LIMITS['KDD'], low_memory=False
)
for column in ['Row', 'Correct First Attempt', 'Hints', 'Incorrects']:
    kdd_raw[column] = pd.to_numeric(kdd_raw[column], errors='coerce')
for column in ['Step Start Time', 'First Transaction Time', 'Step End Time']:
    kdd_raw[column] = pd.to_datetime(kdd_raw[column], errors='coerce')
kdd_raw['event_time'] = (
    kdd_raw['Step Start Time'].fillna(kdd_raw['First Transaction Time']).fillna(kdd_raw['Step End Time'])
)
duplicate_event = kdd_raw['Row'].notna() & kdd_raw.duplicated('Row', keep='first')
kdd_raw = kdd_raw.loc[~duplicate_event]
kdd_raw = kdd_raw[
    kdd_raw['Anon Student Id'].notna() & kdd_raw['Correct First Attempt'].isin([0, 1])
].copy()
kdd_raw['skill_items'] = kdd_raw['KC(Default)'].fillna('').astype(str).apply(
    lambda value: [f'skill:{part.strip()}' for part in value.split('~~') if part.strip()]
)
kdd_raw['problem_item'] = (
    'problem:' + kdd_raw['Problem Hierarchy'].fillna('Unknown').astype(str).str.strip()
    + '::' + kdd_raw['Problem Name'].fillna('Unknown').astype(str).str.strip()
)
kdd_raw['hints'] = kdd_raw['Hints'].clip(lower=0).fillna(0)
kdd_raw['attempts'] = (1 + kdd_raw['Incorrects'].clip(lower=0).fillna(0))
kdd_raw['problem_type'] = 'KDD Algebra problem'
kdd_raw['hierarchy'] = kdd_raw['Problem Hierarchy'].fillna('Unknown').astype(str)
kdd_events = kdd_raw.rename(columns={
    'Anon Student Id': 'learner_id', 'Row': 'event_id',
    'Correct First Attempt': 'correct',
})[[
    'learner_id', 'event_id', 'event_time', 'problem_item', 'skill_items',
    'correct', 'hints', 'attempts', 'problem_type', 'hierarchy',
]].copy()
print('KDD standardised events:', kdd_events.shape)
print('Missing timestamps:', kdd_events['event_time'].isna().sum())
kdd_tables, kdd_summary = build_benchmark(kdd_events, 'KDD', KDD_CLUSTER_DIR)
display(pd.Series(kdd_summary))

KDD standardised events: (809694, 10)
Missing timestamps: 0


,0
Dataset,KDD
RawRows,809694
RawLearners,574
EligibleLearners,565
DiscoveryLearners,452
ValidationLearners,113
EarlyInteractions,566460
FutureInteractions,243152
SkillCandidates,99
ProblemCandidates,855


## Save, manifest and reload verification

Verify that Parquet did not break.

In [8]:
all_benchmarks = {
    'ASSISTments': (assist_tables, assist_summary),
    'KDD': (kdd_tables, kdd_summary),
}
manifest_rows = []
for dataset, (tables, summary) in all_benchmarks.items():
    dataset_dir = OUTPUT_ROOT / dataset.lower()
    dataset_dir.mkdir(parents=True, exist_ok=True)
    for filename, table in tables.items():
        path = dataset_dir / filename
        table.to_parquet(path, index=False, compression='snappy')
        written_rows = parquet_rows(path)
        assert written_rows == len(table)
        manifest_rows.append({
            'Dataset': dataset, 'File': filename, 'Rows': written_rows,
            'Bytes': path.stat().st_size,
        })
    with open(dataset_dir / 'benchmark_summary.json', 'w', encoding='utf-8') as file:
        json.dump(summary, file, indent=2, default=lambda value: value.item() if hasattr(value, 'item') else str(value))

summary_table = pd.DataFrame([assist_summary, kdd_summary])
manifest = pd.DataFrame(manifest_rows)
summary_table.to_csv(OUTPUT_ROOT / 'benchmark_summary.csv', index=False)
manifest.to_csv(OUTPUT_ROOT / 'artifact_manifest.csv', index=False)
benchmark_config = {
    'random_state': RANDOM_STATE,
    'early_fraction': EARLY_FRACTION,
    'validation_fraction': VALIDATION_FRACTION,
    'minimum_interactions': MIN_INTERACTIONS,
    'mastery_prior_strength': MASTERY_PRIOR_STRENGTH,
    'minimum_mastery_evidence': MIN_MASTERY_EVIDENCE,
    'weak_mastery_threshold': WEAK_MASTERY_THRESHOLD,
    'mastered_mastery_threshold': MASTERED_MASTERY_THRESHOLD,
    'catalog_thresholds': CATALOG_THRESHOLDS,
    'candidate_statistics_source': 'discovery_early_only',
    'future_role': 'relevance_labels_only',
    'skill_unknowns_excluded': True,
    'assist_skill_name_mapping_source': 'discovery_early_only',
    'unmapped_named_skill_fallback': 'skill:text:<normalised-name>',
    'relevance_definition': 'at least one observed future interaction',
}
with open(OUTPUT_ROOT / 'benchmark_config.json', 'w', encoding='utf-8') as file:
    json.dump(benchmark_config, file, indent=2)
benchmark_schema = {
    'learner_splits.parquet': 'One cutoff query per eligible learner, cohort, cluster ablation label, cold-start and evaluability fields.',
    'skill_catalog.parquet': 'Training-only skill candidates, support, popularity, difficulty and prior success.',
    'problem_catalog.parquet': 'Training-only problem candidates and content metadata.',
    'early_skill_history.parquet': 'Early learner-skill evidence with empirical-Bayes mastery and skill state.',
    'future_skill_relevance.parquet': 'Future observed skill labels, success alternative, catalogue and seen-before flags.',
    'early_problem_history.parquet': 'Early learner-problem interaction summaries.',
    'future_problem_relevance.parquet': 'Future observed problem labels, success alternative, catalogue and seen-before flags.',
    'problem_skill_map.parquet': 'Training-only problem-to-skill associations for content-based recommendation.',
    'skill_name_id_map.parquet': 'Discovery-early-only ASSISTments name-to-ID mapping; empty for KDD.',
}
with open(OUTPUT_ROOT / 'benchmark_schema.json', 'w', encoding='utf-8') as file:
    json.dump(benchmark_schema, file, indent=2)

assert len(pd.read_csv(OUTPUT_ROOT / 'artifact_manifest.csv')) == len(manifest)
assert len(pd.read_csv(OUTPUT_ROOT / 'benchmark_summary.csv')) == 2
assert set(benchmark_schema) == set(assist_tables) == set(kdd_tables)
display(summary_table)
display(manifest)
print('Saved and verified recommendation benchmark:', OUTPUT_ROOT)

,Dataset,RawRows,RawLearners,EligibleLearners,DiscoveryLearners,ValidationLearners,EarlyInteractions,FutureInteractions,SkillCandidates,ProblemCandidates,...,ValidationProblemEvaluableRate,ValidationSkillColdStartRate,ValidationProblemColdStartRate,ClusterSelectionStatus,TrainingSkillNameMappings,AmbiguousTrainingSkillNames,EarlyRowsMappedFromName,FutureRowsMappedFromName,EarlyTextFallbackRows,FutureTextFallbackRows
0,ASSISTments,6117947,46667,33335,26668,6667,4184236,1814850,161,39779,...,0.899355,0.375731,0.024599,accepted,194,33,0,0,0,0
1,KDD,809694,574,565,452,113,566460,243152,99,855,...,0.982301,0.000000,0.000000,exploratory_fallback,0,0,0,0,0,0


,Dataset,File,Rows,Bytes
0,ASSISTments,learner_splits.parquet,33335,1283461
1,ASSISTments,skill_catalog.parquet,161,16637
2,ASSISTments,problem_catalog.parquet,39779,978772
3,ASSISTments,early_skill_history.parquet,260768,8707997
4,ASSISTments,future_skill_relevance.parquet,174823,4960293
5,ASSISTments,early_problem_history.parquet,4052498,76911901
6,ASSISTments,future_problem_relevance.parquet,1783782,35311927
7,ASSISTments,problem_skill_map.parquet,17826,176578
8,ASSISTments,skill_name_id_map.parquet,194,10524
9,KDD,learner_splits.parquet,565,46435


Saved and verified recommendation benchmark: /content/drive/MyDrive/datasets/recommendation_benchmark_final_outputs
